# 10장 종합 과제 정답: 내 텍스트로 GraphRAG 만들기

실습 가이드 10장 10-2의 종합 과제를 한 가지 텍스트로 풀어 본 예시 답안입니다. 텍스트와 질문은 사람마다 다르므로, 결과를 맞히는 것보다 **방식마다 무엇을 근거로 답했는지** 비교하는 데 씁니다.

| 항목 | 내용 |
|---|---|
| 연결할 DB | 빈 DB (Docker `bolt://localhost:7689`, AuraDB, Blank Sandbox) |
| .env 설정 | 빈 DB의 NEO4J 네 줄 (실습 가이드 5-5) |
| 사용하는 코드 | 7장(지식 그래프, 벡터 RAG), 8장(Text2Cypher), 9장(retrieval_query) |

예시 텍스트는 가상의 회사 소개입니다. 실제 회사가 아니어서 LLM이 사전 지식으로 답을 채울 수 없으므로, 답변이 검색한 내용에서 나왔는지 확인하기 쉽습니다.

## 연결 확인

7장과 같은 빈 DB를 씁니다. Recommendations(Movie)나 POLE(Crime)에 연결되면 연결 셀이 오류로 멈춥니다.

In [1]:
from tools.graphrag_tools import connect_graph

# forbid: 예제 데이터가 든 DB에 연결됐으면 여기서 멈춘다. 아래 셀이 DB를 비우기 때문이다.
graph, NEO4J = connect_graph(expect=[], forbid=["Movie", "Crime"])

연결: bolt://localhost:7689 | DB: neo4j
{'label': '__Entity__', 'n': 16}
{'label': 'Document', 'n': 2}
주의: 빈 DB가 아닙니다. 다른 데이터가 있는 DB에 연결됐는지 .env를 확인하세요.


DB 비우기

7장에서 만든 마리 퀴리 그래프와 섞이지 않도록 비우고 시작합니다(실습 가이드 10-2). 7장의 벡터 인덱스 `document_index`도 지웁니다. 같은 이름의 인덱스를 다시 만들기 때문입니다.

In [2]:
# 모든 노드와 관계를 지운다. 연결 셀이 빈 DB인지 확인한 뒤에만 여기까지 온다(forbid).
graph.query("MATCH (n) DETACH DELETE n")
graph.query("DROP INDEX document_index IF EXISTS")
print("노드 수:", graph.query("MATCH (n) RETURN count(n) AS n")[0]["n"])

노드 수: 0


## 1. 텍스트 고르기와 스키마 설계

관계가 많이 드러나는 4문단짜리 회사 소개를 고릅니다. 질문에 필요한 관계(누가 세웠는가, 어떤 제품을 냈는가, 부품은 누가 공급하는가, 누구와 협력했는가)를 먼저 정하고, 그에 맞춰 `allowed_nodes`, `allowed_relationships`를 정합니다.

| 설계 | 값 | 이유 |
|---|---|---|
| 노드 | Person, Company, Product, Location, Organization, Award | 회사와 일반 기관(대학, 병원)을 나눠 Text2Cypher가 고르기 쉽게 한다 |
| 관계 | FOUNDED, LEADS, WORKS_AT, WORKED_AT, RELEASED, SUPPLIES, USES, LOCATED_IN, PARTNERED_WITH, DEVELOPED_WITH, WON | 질문 세 개에 필요한 관계만 남긴다 |

예시 텍스트 (한국어 번역)

아래 코드의 `my_text`는 영어 원문이고, 내용은 다음과 같습니다. 그래프로 추출될 관계가 드러나는 표현은 굵게 표시했습니다.

> Nimbus Robotics는 2015년 **서하나(Hana Seo)와 대니얼 오르티스(Daniel Ortiz)가 대한민국 부산에서 세운** 로봇 회사다. 서하나는 **최고경영자(CEO)**이고, 대니얼 오르티스는 **연구팀을 이끈다**. 회사는 병원과 물류 창고에서 쓰는 자율 배송 로봇을 만든다.
>
> Nimbus Robotics는 2018년 첫 제품인 **Courier One 로봇을 출시했다**. Courier One은 독일 뮌헨에 있는 **Optiq Sensors가 공급하는** 라이다 센서를 **사용한다**. 2021년에는 **Courier Two를 출시했는데**, 여기에는 **서울대학교와 함께 개발한** 엘리베이터 제어 모듈이 추가됐다.
>
> 회사의 사무소는 세 곳이다. **부산 본사, 싱가포르 영업소, 대전 연구소**다. 박미나(Mina Park)가 **싱가포르 영업소를 관리하고**, 이준(Jun Lee)이 **대전 연구소를 이끈다**. 이준은 이전에 **Optiq Sensors에서 일했다**.
>
> 2023년 Nimbus Robotics는 Courier Two 로봇 40대를 배치하기로 **하늘병원(Haneul Hospital)과 협력 계약을 맺었다**. 병원은 약품 배송 시간이 30% 줄었다고 밝혔다. 같은 해 Nimbus Robotics는 **아시아 로봇 혁신상(Asia Robotics Innovation Award)을 받았다**.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_experimental.graph_transformers import LLMGraphTransformer

my_text = """
Nimbus Robotics is a robotics company founded in 2015 by Hana Seo and Daniel Ortiz in Busan, South Korea.
Hana Seo is the chief executive officer, and Daniel Ortiz leads the research team.
The company builds autonomous delivery robots for hospitals and warehouses.

Nimbus Robotics released its first product, the Courier One robot, in 2018.
Courier One uses a lidar sensor supplied by Optiq Sensors, a company based in Munich, Germany.
In 2021 Nimbus Robotics released Courier Two, which added an elevator-control module developed together with Seoul National University.

The company has three offices: the headquarters in Busan, a sales office in Singapore, and a research lab in Daejeon.
Mina Park manages the Singapore office, and Jun Lee leads the Daejeon lab. Jun Lee previously worked at Optiq Sensors.

In 2023 Nimbus Robotics signed a partnership with Haneul Hospital to deploy 40 Courier Two robots.
The hospital reported that delivery times for medicine fell by 30 percent.
The same year, Nimbus Robotics received the Asia Robotics Innovation Award.
"""

# 7장과 같은 방법. 문단이 4개라 청크 크기를 400자로 잡아 문단이 대체로 한 청크에 들어가게 한다.
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
documents = splitter.create_documents([my_text])
print("청크 수:", len(documents))

llm = ChatOpenAI(model="gpt-4o", temperature=0)
# 허용 목록은 텍스트와 질문에 맞게 직접 정한다(과제 2단계).
transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=["Person", "Company", "Product", "Location", "Organization", "Award"],
    allowed_relationships=["FOUNDED", "LEADS", "WORKS_AT", "WORKED_AT", "RELEASED", "SUPPLIES",
                           "USES", "LOCATED_IN", "PARTNERED_WITH", "DEVELOPED_WITH", "WON"],
)
graph_documents = transformer.convert_to_graph_documents(documents)
graph.add_graph_documents(graph_documents, baseEntityLabel=True, include_source=True)

# 무엇이 만들어졌는지 레이블과 관계로 확인한다.
for row in graph.query("MATCH (n) RETURN [l IN labels(n) WHERE l <> '__Entity__'][0] AS label, count(*) AS n ORDER BY n DESC"):
    print(row)
for row in graph.query("""
MATCH (a)-[r]->(b) WHERE NOT a:Document
RETURN a.id AS source, type(r) AS rel, b.id AS target ORDER BY rel, source
"""):
    print(f"  {row['source']} -[{row['rel']}]-> {row['target']}")

청크 수: 4
{'label': 'Location', 'n': 5}
{'label': 'Person', 'n': 4}
{'label': 'Document', 'n': 4}
{'label': 'Company', 'n': 3}
{'label': 'Product', 'n': 2}
{'label': 'Organization', 'n': 2}
{'label': 'Award', 'n': 1}
  Courier Two -[DEVELOPED_WITH]-> Seoul National University
  Nimbus Robotics -[FOUNDED]-> Hana Seo
  Nimbus Robotics -[FOUNDED]-> Daniel Ortiz
  Daniel Ortiz -[LEADS]-> Nimbus Robotics
  Hana Seo -[LEADS]-> Nimbus Robotics
  Jun Lee -[LEADS]-> Daejeon
  Mina Park -[LEADS]-> Singapore
  Company -[LOCATED_IN]-> Daejeon
  Company -[LOCATED_IN]-> Singapore
  Company -[LOCATED_IN]-> Busan
  Nimbus Robotics -[LOCATED_IN]-> Busan, South Korea
  Optiq Sensors -[LOCATED_IN]-> Munich
  Nimbus Robotics -[PARTNERED_WITH]-> Haneul Hospital
  Nimbus Robotics -[RELEASED]-> Courier Two
  Nimbus Robotics -[RELEASED]-> Courier One
  Courier One -[USES]-> Optiq Sensors
  Nimbus Robotics -[WON]-> Asia Robotics Innovation Award
  Jun Lee -[WORKED_AT]-> Optiq Sensors


Neo4j Browser에서도 확인합니다(Docker 빈 DB는 http://localhost:7476).

```cypher
MATCH (d:Document)-[:MENTIONS]->(e)-[r]->(e2)
RETURN d, e, r, e2 LIMIT 80
```

## 2. 벡터 RAG: 내용 요약형 질문

7장과 같은 방법으로 `Document` 노드를 임베딩하고 RAG 체인을 만듭니다. 질문은 원문 언어(영어)로 합니다. 7-3에서 본 것처럼 한국어 질문과 영어 문서는 유사도가 낮게 나오기 때문입니다.

In [4]:
from langchain_openai import OpenAIEmbeddings
from langchain_neo4j import Neo4jVector
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = Neo4jVector.from_existing_graph(
    embedding=embeddings,
    **NEO4J,
    index_name="document_index",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding",
)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

prompt = ChatPromptTemplate.from_messages([
    ("system", "주어진 문서만 근거로 한국어로 간결하게 답한다. 문서에 없으면 '정보가 부족합니다'라고 답한다."),
    ("human", "문서:\n{context}\n\n질문: {question}"),
])

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

q_vector = "What does Nimbus Robotics build, and what result did the hospital deployment show?"
for doc in retriever.invoke(q_vector):
    print("검색:", doc.page_content.replace("\n", " ")[:100])
print("답변:", rag_chain.invoke(q_vector))

검색:  text: In 2023 Nimbus Robotics signed a partnership with Haneul Hospital to deploy 40 Courier Two ro
검색:  text: Nimbus Robotics released its first product, the Courier One robot, in 2018. Courier One uses 
답변: Nimbus Robotics는 Courier Two 로봇을 제작하며, 병원 배치 결과 약품 배송 시간이 30% 감소했습니다.


## 3. Text2Cypher: 개수·목록형 질문

새 그래프의 스키마를 다시 읽은 뒤 8장의 `GraphCypherQAChain`으로 질문합니다. 생성된 Cypher와 조회 결과를 함께 출력하고, 같은 질문의 기준 쿼리를 직접 실행해 정답 여부를 판단합니다.

In [5]:
from langchain_neo4j import GraphCypherQAChain

# 적재 뒤 스키마가 바뀌었으므로 다시 읽는다. 빠뜨리면 LLM이 예전 스키마를 보고 쿼리를 만든다.
graph.refresh_schema()
print(graph.schema)

cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    validate_cypher=True,
    return_intermediate_steps=True,
    allow_dangerous_requests=True,
)

q_cypher = "How many products has Nimbus Robotics released? List their names."
result = cypher_chain.invoke({"query": q_cypher})
print("생성된 Cypher:\n", result["intermediate_steps"][0]["query"])
print("조회 결과:", result["intermediate_steps"][1]["context"])
print("답변:", result["result"])

# 정답 확인용 기준 쿼리. 관계 이름과 방향은 1절 적재 결과에서 확인한 것을 쓴다.
print("기준 쿼리:", graph.query("""
MATCH (:Company {id: 'Nimbus Robotics'})-[:RELEASED]->(p:Product)
RETURN count(p) AS products, collect(p.id) AS names
"""))

Node properties:
Document {text: STRING, embedding: LIST, id: STRING}
Person {id: STRING}
Award {id: STRING}
Organization {id: STRING}
Company {id: STRING}
Location {id: STRING}
Product {id: STRING}
Relationship properties:

The relationships:
(:Document)-[:MENTIONS]->(:Organization)
(:Document)-[:MENTIONS]->(:Product)
(:Document)-[:MENTIONS]->(:Location)
(:Document)-[:MENTIONS]->(:Company)
(:Document)-[:MENTIONS]->(:Person)
(:Document)-[:MENTIONS]->(:Award)
(:Person)-[:LEADS]->(:Company)
(:Person)-[:LEADS]->(:Location)
(:Person)-[:WORKED_AT]->(:Company)
(:Company)-[:LOCATED_IN]->(:Location)
(:Company)-[:WON]->(:Award)
(:Company)-[:FOUNDED]->(:Person)
(:Company)-[:RELEASED]->(:Product)
(:Company)-[:PARTNERED_WITH]->(:Organization)
(:Product)-[:USES]->(:Company)
(:Product)-[:DEVELOPED_WITH]->(:Organization)
생성된 Cypher:
 MATCH (c:Company {id: 'Nimbus Robotics'})-[:RELEASED]->(p:Product)
RETURN COUNT(p) AS numberOfProducts, COLLECT(p.id) AS productNames

조회 결과: [{'numberOfProducts': 2, 'p

## 4. 하이브리드: 내용과 관계를 함께 묻는 질문

9장과 같은 방식으로 `retrieval_query`를 씁니다. 다만 이번 그래프에서 원문은 `Document` 노드에 있고, 엔티티와는 `(Document)-[:MENTIONS]->(엔티티)`로 이어집니다(실습 가이드 10-2 "막히는 지점"). 그래서 벡터 검색으로 찾은 `Document`에서 `MENTIONS`로 엔티티에 가고, 그 엔티티의 관계를 한 번 더 따라가 사실 목록(`facts`)을 만듭니다.

In [6]:
retrieval_query = """
WITH node, score
OPTIONAL MATCH (node)-[:MENTIONS]->(e)-[r]-(other)
WHERE NOT other:Document
WITH node, score,
     CASE WHEN r IS NULL THEN null
          WHEN startNode(r) = e THEN e.id + ' -[' + type(r) + ']-> ' + other.id
          ELSE other.id + ' -[' + type(r) + ']-> ' + e.id END AS fact
WITH node, score, collect(DISTINCT fact) AS facts
RETURN node.text AS text, score, {facts: facts[..40]} AS metadata
"""

# 7장에서 만든 인덱스에 연결만 한다. retrieval_query가 Document에서 그래프를 한 단계 더 탐색한다.
hybrid_store = Neo4jVector.from_existing_index(
    embedding=embeddings,
    index_name="document_index",
    **NEO4J,
    retrieval_query=retrieval_query,
)
hybrid_retriever = hybrid_store.as_retriever(search_kwargs={"k": 2})

def format_hybrid(docs):
    return "\n\n".join(
        f"원문: {d.page_content}\n그래프 관계:\n" + "\n".join(d.metadata["facts"])
        for d in docs
    )

hybrid_prompt = ChatPromptTemplate.from_messages([
    ("system", "원문과 그래프 관계를 근거로 한국어로 답한다. 답에 쓴 관계는 그래프 관계 목록에서 찾아 근거로 함께 적는다. 원문과 관계 어디에도 없는 내용은 쓰지 않고, 그럴 때는 '정보가 부족합니다'라고 답한다."),
    ("human", "문맥:\n{context}\n\n질문: {question}"),
])
hybrid_chain = (
    {"context": hybrid_retriever | format_hybrid, "question": RunnablePassthrough()}
    | hybrid_prompt
    | llm
    | StrOutputParser()
)

q_hybrid = "Which organizations helped build the Courier robots, and how were they involved?"
for d in hybrid_retriever.invoke(q_hybrid):
    print("검색:", d.page_content.replace("\n", " ")[:90])
    for fact in d.metadata["facts"]:
        print("  관계:", fact)
print("답변:", hybrid_chain.invoke(q_hybrid))

# 같은 질문을 벡터 RAG에 넣어 비교한다. 관계 목록이 없으면 무엇이 달라지는가?
print("\n[비교] 벡터 RAG 답변:", rag_chain.invoke(q_hybrid))

검색: Nimbus Robotics released its first product, the Courier One robot, in 2018. Courier One us
  관계: Courier Two -[DEVELOPED_WITH]-> Seoul National University
  관계: Nimbus Robotics -[RELEASED]-> Courier Two
  관계: Optiq Sensors -[LOCATED_IN]-> Munich
  관계: Jun Lee -[WORKED_AT]-> Optiq Sensors
  관계: Courier One -[USES]-> Optiq Sensors
  관계: Nimbus Robotics -[RELEASED]-> Courier One
  관계: Nimbus Robotics -[WON]-> Asia Robotics Innovation Award
  관계: Nimbus Robotics -[PARTNERED_WITH]-> Haneul Hospital
  관계: Nimbus Robotics -[FOUNDED]-> Hana Seo
  관계: Daniel Ortiz -[LEADS]-> Nimbus Robotics
  관계: Hana Seo -[LEADS]-> Nimbus Robotics
  관계: Nimbus Robotics -[LOCATED_IN]-> Busan, South Korea
  관계: Nimbus Robotics -[FOUNDED]-> Daniel Ortiz
검색: In 2023 Nimbus Robotics signed a partnership with Haneul Hospital to deploy 40 Courier Two
  관계: Nimbus Robotics -[WON]-> Asia Robotics Innovation Award
  관계: Courier Two -[DEVELOPED_WITH]-> Seoul National University
  관계: Nimbus Robotics -[RELEASED]-> Cou

## 5. 결과 기록 (제출물)

실행 결과 (2026-11-04, Docker Neo4j 5.26 빈 DB, gpt-4o, temperature 0)

| 방식 | 질문 | 생성된 Cypher | 답변 | 정답 여부 |
|---|---|---|---|---|
| 벡터 RAG | What does Nimbus Robotics build, and what result did the hospital deployment show? | (없음) | Nimbus Robotics는 Courier Two 로봇을 제작하며, 병원 배치 결과 약품 배송 시간이 30% 감소했습니다. | 부분 정답. 30% 감소는 맞지만 "병원·창고용 자율 배송 로봇"을 만든다는 첫 문단이 검색되지 않았다 |
| Text2Cypher | How many products has Nimbus Robotics released? List their names. | `MATCH (c:Company {id: 'Nimbus Robotics'})-[:RELEASED]->(p:Product) RETURN COUNT(p), COLLECT(p.id)` | 2개, Courier One과 Courier Two (영어로 답함) | 정답. 기준 쿼리 결과와 같다 |
| 하이브리드 | Which organizations helped build the Courier robots, and how were they involved? | (`retrieval_query`로 고정) | Courier One은 Optiq Sensors의 라이다 센서를 사용, Courier Two는 서울대와 엘리베이터 제어 모듈을 공동 개발. 근거 관계 `USES`, `DEVELOPED_WITH`를 함께 표기 | 정답 |

내 데이터에는 어떤 방식이 맞았는가

이 텍스트처럼 짧고 문단마다 사실이 모여 있는 글은 벡터 RAG만으로도 대부분의 질문에 답할 수 있었다. 하이브리드 질문에도 벡터 RAG가 같은 내용으로 답했다. 하지만 "제품이 몇 개인가"처럼 여러 문단에 흩어진 항목을 세는 질문은 Text2Cypher가 DB에서 정확히 계산했고, 하이브리드는 답의 근거가 된 관계를 함께 보여 줘서 답을 검증하기 쉬웠다. 반대로 Text2Cypher와 하이브리드의 정확도는 그래프 품질에 달려 있었다. 아래 해설처럼 추출된 그래프에 방향이 틀린 관계와 뭉뚱그려진 노드가 있어서, 질문이 그 부분을 건드렸다면 오류 없이 틀린 답이 나왔을 것이다.

### 해설

**1. 추출된 그래프의 품질 (평가 기준: 스키마 설계)**

1절 출력의 관계 목록을 원문과 대조하면 다음 문제가 보입니다.

| 추출 결과 | 원문 | 문제 |
|---|---|---|
| `Nimbus Robotics -[FOUNDED]-> Hana Seo` | Hana Seo가 회사를 세웠다 | 방향이 반대다. "누가 세웠나"를 Text2Cypher로 물으면 패턴 방향에 따라 0행이 나올 수 있다 |
| `Company -[LOCATED_IN]-> Busan / Singapore / Daejeon` | 본사, 영업소, 연구소 3곳 | id가 "Company"인 노드 하나에 사무소 3곳이 묶였다. 사무소를 나타낼 `Office` 노드가 허용 목록에 없었기 때문이다 |
| `Jun Lee -[LEADS]-> Daejeon` | Jun Lee는 대전 연구소를 이끈다 | 같은 이유로 연구소가 아니라 도시에 연결됐다 |
| `Courier One -[USES]-> Optiq Sensors` | Optiq Sensors가 라이다 센서를 공급한다 | `SUPPLIES`를 허용했지만 쓰이지 않았고, 센서라는 정보가 사라졌다 |
| Hana Seo, Daniel Ortiz 모두 `LEADS` | CEO, 연구팀장 | 직책 차이가 관계에 남지 않는다 |
| (없음) | 40대 배치, 30% 감소, 연도 | 수치와 연도는 노드 속성으로 추출되지 않고 원문에만 남았다 |

고치는 방법: `allowed_nodes`에 `Office`를 넣고, 관계를 `(Person, FOUNDED, Company)`처럼 방향까지 정한 튜플로 지정하거나, `node_properties`로 연도·수량을 속성으로 뽑게 한다. 고친 뒤에는 다시 적재하고 1절의 관계 목록을 원문과 다시 대조한다.

**2. 벡터 RAG가 부분 정답이 된 이유**

`k=2`로 검색해 병원 문단과 제품 문단만 들어갔습니다. 회사가 무엇을 만드는지는 첫 문단에 있었는데, 질문에 "hospital deployment"가 들어 있어 병원 문단이 1위가 됐습니다. LLM은 받은 문맥 안에서 Courier Two를 "만드는 것"으로 답했습니다. 유사도 점수는 병원 문단 0.864, 제품 문단 0.817, 회사 소개 문단 0.815였습니다. 2위와 3위의 차이가 0.002라서 `k=2`의 경계에서 답이 든 청크가 잘렸습니다. `k=3`으로 늘리면 첫 문단이 들어갑니다. 7장 과제 3의 `k` 비교와 같은 현상입니다.

**3. Text2Cypher가 맞은 이유와 답변 언어**

질문이 깨끗하게 추출된 `RELEASED` 관계 하나로 풀리는 질문이었습니다. 1번 표의 `FOUNDED`처럼 방향이 틀린 관계를 쓰는 질문이었다면 결과가 달랐을 수 있으므로, 기준 쿼리를 직접 실행해 확인하는 단계가 필요합니다. 답변이 영어로 나온 것은 `GraphCypherQAChain`의 기본 답변 프롬프트(`qa_prompt`)에 언어 지정이 없기 때문입니다. 한국어 답이 필요하면 8장처럼 `qa_prompt`를 직접 넣습니다.

**4. 하이브리드 프롬프트 한 줄의 차이**

처음에는 시스템 프롬프트를 "관계는 그래프 관계 목록에 있는 것만 쓰고, 없으면 '정보가 부족합니다'라고 답한다"로 실행했습니다. 그러자 같은 문맥(`USES`, `DEVELOPED_WITH`가 들어 있음)에서도 "정보가 부족합니다"라고 답했습니다(2026-11-04). 질문의 "helped build"에 해당하는 관계 이름이 목록에 없어서 LLM이 근거가 없다고 판단한 것입니다. "답에 쓴 관계를 목록에서 찾아 근거로 함께 적는다"로 바꾸자 위 표처럼 근거와 함께 맞게 답했습니다. 9장 과제 3과 같은 교훈입니다. 프롬프트의 제약은 환각을 막지만, 너무 엄격하면 답할 수 있는 질문도 거절합니다.

**5. 하이브리드의 `facts`에 검색되지 않은 청크의 관계도 들어간 이유**

`retrieval_query`는 검색된 `Document`가 언급한 엔티티에서 관계를 한 단계 더 따라갑니다. 그래서 제품 문단이 언급한 Optiq Sensors를 통해, 다른 문단에 있던 `Jun Lee -[WORKED_AT]-> Optiq Sensors`까지 문맥에 들어왔습니다. 벡터 검색만으로는 얻지 못하는, 청크 경계를 넘는 정보입니다. 대신 문맥이 길어지므로 `facts[..40]`처럼 개수를 제한합니다.

## 6. 한글 텍스트로 했을 때

같은 내용을 한국어 번역문(1절 위의 번역)으로 바꿔 지식 그래프를 다시 만들고, Text2Cypher에 한국어로 질문합니다. 허용 목록과 청크 설정은 1절과 같습니다. DB를 다시 비우므로 1~4절의 영어 그래프는 지워집니다.

In [7]:
graph.query("MATCH (n) DETACH DELETE n")
graph.query("DROP INDEX document_index IF EXISTS")

ko_text = """
Nimbus Robotics는 2015년 서하나(Hana Seo)와 대니얼 오르티스(Daniel Ortiz)가 대한민국 부산에서 세운 로봇 회사다. 서하나는 최고경영자(CEO)이고, 대니얼 오르티스는 연구팀을 이끈다. 회사는 병원과 물류 창고에서 쓰는 자율 배송 로봇을 만든다.

Nimbus Robotics는 2018년 첫 제품인 Courier One 로봇을 출시했다. Courier One은 독일 뮌헨에 있는 Optiq Sensors가 공급하는 라이다 센서를 사용한다. 2021년에는 Courier Two를 출시했는데, 여기에는 서울대학교와 함께 개발한 엘리베이터 제어 모듈이 추가됐다.

회사의 사무소는 세 곳이다. 부산 본사, 싱가포르 영업소, 대전 연구소다. 박미나(Mina Park)가 싱가포르 영업소를 관리하고, 이준(Jun Lee)이 대전 연구소를 이끈다. 이준은 이전에 Optiq Sensors에서 일했다.

2023년 Nimbus Robotics는 Courier Two 로봇 40대를 배치하기로 하늘병원(Haneul Hospital)과 협력 계약을 맺었다. 병원은 약품 배송 시간이 30% 줄었다고 밝혔다. 같은 해 Nimbus Robotics는 아시아 로봇 혁신상(Asia Robotics Innovation Award)을 받았다.
"""

# 1절과 같은 splitter, transformer를 그대로 쓴다.
ko_documents = splitter.create_documents([ko_text])
print("청크 수:", len(ko_documents), "| 청크별 글자 수:", [len(d.page_content) for d in ko_documents])
graph.add_graph_documents(transformer.convert_to_graph_documents(ko_documents), baseEntityLabel=True, include_source=True)

# 노드 id가 어떤 언어로 저장됐는지 본다.
print("[노드]")
for row in graph.query("MATCH (n:__Entity__) RETURN [l IN labels(n) WHERE l <> '__Entity__'][0] AS label, n.id AS id ORDER BY label, id"):
    print(f"  {row['label']:12} {row['id']}")
print("[관계]")
for row in graph.query("MATCH (a)-[r]->(b) WHERE NOT a:Document RETURN a.id AS s, type(r) AS rel, b.id AS t ORDER BY rel, s"):
    print(f"  {row['s']} -[{row['rel']}]-> {row['t']}")

청크 수: 2 | 청크별 글자 수: [330, 312]
[노드]
  Award        아시아 로봇 혁신상
  Company      Nimbus Robotics
  Company      Optiq Sensors
  Location     Busan, South Korea
  Location     Munich, Germany
  Location     대전 연구소
  Location     부산 본사
  Location     싱가포르 영업소
  Organization Seoul National University
  Organization 하늘병원
  Person       Daniel Ortiz
  Person       Hana Seo
  Person       박미나
  Person       이준
  Product      Courier One
  Product      Courier Two
[관계]
  Courier Two -[DEVELOPED_WITH]-> Seoul National University
  Nimbus Robotics -[FOUNDED]-> Daniel Ortiz
  Nimbus Robotics -[FOUNDED]-> Hana Seo
  Daniel Ortiz -[LEADS]-> Nimbus Robotics
  Hana Seo -[LEADS]-> Nimbus Robotics
  박미나 -[LEADS]-> 싱가포르 영업소
  이준 -[LEADS]-> 대전 연구소
  Nimbus Robotics -[LOCATED_IN]-> Busan, South Korea
  Optiq Sensors -[LOCATED_IN]-> Munich, Germany
  Nimbus Robotics -[PARTNERED_WITH]-> 하늘병원
  Nimbus Robotics -[RELEASED]-> Courier Two
  Nimbus Robotics -[RELEASED]-> Courier One
  Courier One -[USES]-> Optiq Se

In [8]:
# 한국어 질문으로 Text2Cypher를 실행한다. LLM이 질문의 이름을 그대로 id로 쓰는지 본다.
graph.refresh_schema()
ko_chain = GraphCypherQAChain.from_llm(
    llm=llm, graph=graph, validate_cypher=True,
    return_intermediate_steps=True, allow_dangerous_requests=True,
)

ko_questions = [
    "Nimbus Robotics가 출시한 제품은 몇 개이고 이름은 무엇인가요?",
    "님버스 로보틱스가 출시한 제품은 몇 개인가요?",
    "서하나가 세운 회사는 어디인가요?",
    "이준이 이전에 일한 회사는 어디인가요?",
]
for q in ko_questions:
    r = ko_chain.invoke({"query": q})
    print("질문:", q)
    print("  Cypher:", r["intermediate_steps"][0]["query"].replace("\n", " ").strip())
    print("  결과:", r["intermediate_steps"][1]["context"])
    print("  답변:", r["result"])

질문: Nimbus Robotics가 출시한 제품은 몇 개이고 이름은 무엇인가요?
  Cypher: MATCH (c:Company {id: "Nimbus Robotics"})-[:RELEASED]->(p:Product) RETURN count(p) AS productCount, collect(p.id) AS productNames
  결과: [{'productCount': 2, 'productNames': ['Courier One', 'Courier Two']}]
  답변: Nimbus Robotics가 출시한 제품은 2개이며, 이름은 Courier One, Courier Two입니다.
질문: 님버스 로보틱스가 출시한 제품은 몇 개인가요?
  Cypher: MATCH (c:Company {id: "님버스 로보틱스"})-[:RELEASED]->(p:Product) RETURN count(p) AS numberOfProducts
  결과: [{'numberOfProducts': 0}]
  답변: 님버스 로보틱스가 출시한 제품의 수를 알 수 없습니다.
질문: 서하나가 세운 회사는 어디인가요?
  Cypher: MATCH (p:Person {id: "서하나"})<-[:FOUNDED]-(c:Company) RETURN c.id
  결과: []
  답변: I don't know the answer.
질문: 이준이 이전에 일한 회사는 어디인가요?
  Cypher: MATCH (p:Person {id: "이준"})-[:WORKED_AT]->(c:Company) RETURN c.id
  결과: [{'c.id': 'Optiq Sensors'}]
  답변: 이준이 이전에 일한 회사는 Optiq Sensors입니다.


In [9]:
from langchain_neo4j import Neo4jGraph

# enhanced_schema=True: 스키마에 속성의 실제 값 예시(Available options)를 함께 넣는다.
# LLM이 DB에 저장된 id 표기를 보고 쿼리를 쓸 수 있다.
enhanced_graph = Neo4jGraph(**NEO4J, enhanced_schema=True)
for line in enhanced_graph.schema.splitlines():
    if "Available options" in line and "Document" not in line and "`text`" not in line:
        print(line.strip())

enhanced_chain = GraphCypherQAChain.from_llm(
    llm=llm, graph=enhanced_graph, validate_cypher=True,
    return_intermediate_steps=True, allow_dangerous_requests=True,
)
for q in ["님버스 로보틱스가 출시한 제품은 몇 개인가요?", "서하나가 세운 회사는 어디인가요?"]:
    r = enhanced_chain.invoke({"query": q})
    print("질문:", q)
    print("  Cypher:", r["intermediate_steps"][0]["query"].replace("\n", " ").strip())
    print("  결과:", r["intermediate_steps"][1]["context"])
    print("  답변:", r["result"])

- `id`: STRING Available options: ['fe24ac5be139b98d64b94d4b0275a89f', 'c573aed727cf1347fe5f4a063599366e']
- `id`: STRING Available options: ['Hana Seo', 'Daniel Ortiz', '박미나', '이준']
- `id`: STRING Available options: ['아시아 로봇 혁신상']
- `id`: STRING Available options: ['Seoul National University', '하늘병원']
- `id`: STRING Available options: ['Nimbus Robotics', 'Optiq Sensors']
- `id`: STRING Available options: ['Busan, South Korea', 'Munich, Germany', '부산 본사', '싱가포르 영업소', '대전 연구소']
- `id`: STRING Available options: ['Courier One', 'Courier Two']
질문: 님버스 로보틱스가 출시한 제품은 몇 개인가요?
  Cypher: MATCH (c:Company {id: 'Nimbus Robotics'})-[:RELEASED]->(p:Product) RETURN count(p) AS numberOfProducts
  결과: [{'numberOfProducts': 2}]
  답변: 님버스 로보틱스가 출시한 제품은 2개입니다.
질문: 서하나가 세운 회사는 어디인가요?
  Cypher: MATCH (p:Person {id: 'Hana Seo'})<-[:FOUNDED]-(c:Company) RETURN c.id
  결과: [{'c.id': 'Nimbus Robotics'}]
  답변: 서하나가 세운 회사는 Nimbus Robotics입니다.


### 6절 해설

실행 결과 (2026-11-04, 1절과 같은 허용 목록·청크 설정)

**1. 같은 400자에 내용이 더 많이 들어간다**

영어 원문은 청크 4개였지만 한국어 번역은 청크 2개(330자, 312자)가 됐습니다. `chunk_size`는 글자 수 기준이고, 한글 한 글자가 영어보다 많은 내용을 담기 때문입니다. 문단 두 개가 한 청크에 들어가므로 벡터 검색의 단위가 거칠어집니다. 한글 텍스트는 `chunk_size`를 더 작게 잡습니다.

**2. 노드 id의 언어가 규칙 없이 섞인다**

| 원문 표기 | 저장된 id |
|---|---|
| 서하나(Hana Seo), 대니얼 오르티스(Daniel Ortiz) | `Hana Seo`, `Daniel Ortiz` (영어) |
| 박미나(Mina Park), 이준(Jun Lee), 하늘병원(Haneul Hospital) | `박미나`, `이준`, `하늘병원` (한글) |
| 서울대학교 (영어 병기 없음) | `Seoul National University` (LLM이 번역) |
| 아시아 로봇 혁신상(Asia Robotics Innovation Award) | `아시아 로봇 혁신상` (한글) |

같은 방식으로 영어를 함께 적었는데도 어떤 이름은 영어로, 어떤 이름은 한글로 저장됐습니다. 영어를 적지 않은 서울대학교는 오히려 영어로 바뀌었습니다. 사용자가 어떤 표기로 질문할지 예측할 수 없게 됩니다.

**3. Text2Cypher는 질문의 표기를 그대로 id로 쓴다**

| 질문 | 쿼리의 id | 결과 | 답변 |
|---|---|---|---|
| Nimbus Robotics가 출시한 제품은? | `Nimbus Robotics` | 2개 | 정답 |
| 님버스 로보틱스가 출시한 제품은? | `님버스 로보틱스` | 0 | "알 수 없습니다" |
| 서하나가 세운 회사는? | `서하나` | 0행 | "I don't know the answer." |
| 이준이 이전에 일한 회사는? | `이준` | Optiq Sensors | 정답 |

`graph.schema`에는 레이블과 속성 이름만 있고 실제 값이 없습니다. 그래서 LLM은 질문에 적힌 "님버스 로보틱스", "서하나"를 그대로 id 조건에 넣었고, 쿼리는 오류 없이 0행을 돌려줬습니다. DB의 표기와 질문의 표기가 우연히 같은 "이준"만 맞았습니다. 개수 질문에서 0을 "알 수 없다"로 바꿔 답한 것, 빈 결과에 영어로 답한 것도 함께 볼 점입니다(기본 `qa_prompt`).

**4. `enhanced_schema=True`로 해결된다**

`Neo4jGraph(..., enhanced_schema=True)`는 스키마에 속성의 실제 값 예시(`Available options`)를 함께 넣습니다. LLM이 `Hana Seo`, `Nimbus Robotics` 같은 저장된 표기를 보고 "서하나" → `Hana Seo`, "님버스 로보틱스" → `Nimbus Robotics`로 바꿔 써서 두 질문 모두 맞았습니다. 다만 값이 많은 속성은 일부 예시만 들어가므로, 노드가 많은 그래프에서는 이것만으로 충분하지 않습니다. 근본적으로는 적재할 때 표기를 하나로 정하고(엔티티 정규화), 별칭을 속성으로 저장해 두는 것이 맞습니다.

**5. 좋아진 점과 실행마다 달라지는 점**

- 영어판에서 id "Company" 하나로 묶였던 사무소가 한국어판에서는 `부산 본사`, `싱가포르 영업소`, `대전 연구소` 노드로 따로 추출됐고, `이준 -[LEADS]-> 대전 연구소`도 맞게 연결됐습니다. 언어를 바꾸면 추출 품질이 좋아지기도 나빠지기도 합니다.
- `FOUNDED`는 이번 실행에서 `Nimbus Robotics -[FOUNDED]-> Hana Seo`(방향 반대)로 추출됐습니다. 같은 텍스트를 노트북 밖에서 따로 실행했을 때는 `Hana Seo -[FOUNDED]-> Nimbus Robotics`로 맞게 나왔습니다. temperature 0이어도 추출 결과는 실행마다 달라질 수 있으므로, 적재 후 관계 목록을 원문과 대조하는 단계를 빼지 않습니다. 이번에는 LLM이 스키마의 방향(`(:Company)-[:FOUNDED]->(:Person)`)을 보고 `<-[:FOUNDED]-`로 써서 답은 맞았습니다.

**정리: 한글 텍스트로 할 때의 체크리스트**

1. 이름 표기를 한 언어로 통일한다. 병기하면 LLM이 무엇을 id로 쓸지 정해지지 않는다.
2. `chunk_size`를 영어보다 작게 잡는다.
3. Text2Cypher는 `enhanced_schema=True`를 쓰거나, few-shot 예시에 실제 id 표기를 보여 준다.
4. `qa_prompt`에 "한국어로 답하고, 결과가 0이면 0이라고 답한다"를 넣는다.
5. 질문과 텍스트의 언어를 맞춘다. 벡터 검색은 언어가 같을 때 유사도가 높다(7-3).